# صَوْتَك — تجربة Habibi IRQ على Google Colab

هاي التجربة تشتغل بالمتصفح. اختار **Runtime → Change runtime type → GPU** إذا كان متاحاً، وبعدها شغّل أول خلية ثم الثانية. Google Colab مجاني ضمن حصص غير مضمونة؛ النموذج يحتاج تنزيل كبير وذاكرة كافية.



In [ ]:
%pip -q install habibi-tts
!which habibi-tts_infer-cli


In [ ]:
from pathlib import Path
import re, subprocess, tempfile
from IPython.display import Audio, display
from google.colab import files

script = "هلا والله طلاب، شلونكم؟ اليوم راح نحچي عن موضوع مهم بالكيمياء، ونشرحه وحدة وحدة." #@param {type:"string"}
assert script.strip(), "اكتب النص أولاً"
assert len(script) <= 1600, "بهذه التجربة الحد 1600 حرف حتى ما تصير ثقيلة"

def split_iraqi(text, maximum=160):
    words = re.sub(r"\s+", " ", text.strip()).split(" ")
    out, chunk = [], ""
    for word in words:
        if len(word) > maximum: raise ValueError("كلمة طويلة جداً")
        if len(chunk) + len(word) + bool(chunk) > maximum:
            out.append(chunk)
            chunk = ""
        chunk += (" " if chunk else "") + word
        if len(chunk) >= 65 and re.search(r"[.!؟؛…:]$", word):
            out.append(chunk)
            chunk = ""
    if chunk: out.append(chunk)
    return out

chunks = split_iraqi(script)
print(f"عدد المقاطع: {len(chunks)} — تحميل أوزان النموذج مرة واحدة")
with tempfile.TemporaryDirectory() as tmp:
    source = Path(tmp)/"chunks.txt"
    source.write_text("\n".join("[main] "+s for s in chunks), encoding="utf-8")
    command = [
        "habibi-tts_infer-cli", "--model", "Specialized", "--dialect", "IRQ",
        "--ref_audio", "assets/IRQ.wav",
        "--ref_text", "يعني ااا ما نقدر ناخذ وقت أكثر، ااا لأنه شروط كلش يحتاجلها وقت.",
        "--gen_file", str(source), "--output_dir", tmp, "--output_file", "iraqi.wav"
    ]
    subprocess.run(command, check=True, timeout=1200)
    wav = Path(tmp)/"iraqi.wav"
    assert wav.is_file() and wav.stat().st_size > 100, "ما انولد ملف صوت"
    display(Audio(filename=str(wav)))
    destination = Path("/content/sawtak_habibi_iraqi.wav")
    destination.write_bytes(wav.read_bytes())
    files.download(str(destination))

